### Notebook for processing the validation of the Futhark results

In [48]:
import ast
import re
import numpy as np
import pandas as pd
from pathlib import Path

##### Functions for processing Matlab and Futhark Prices

In [49]:
FUT_TYPE_SUFFIX = re.compile(r'(f16|f32|f64|i8|i16|i32|i64|u8|u16|u32|u64)\b')

def parse_futhark_prices(path):
    """Parse the price matrix from line 1 of a Futhark validate_solve .val file.

    The first line is a Futhark literal of shape [c][Ax], e.g.
    `[[200.0f64, 169.6f64, ...], [260.0f64, ...], ...]`.
    Returns a numpy.ndarray of shape (c, Ax) with dtype float64.
    """
    with open(path) as f:
        first_line = f.readline()
    cleaned = FUT_TYPE_SUFFIX.sub('', first_line)
    nested = ast.literal_eval(cleaned)
    return np.asarray(nested, dtype=np.float64)

def parse_matlab_prices(path):
    """Parse the price matrix from a MATLAB validate_run_illustrations_test .dat file.

    The file is a CSV (one row per car type, columns = ages 0..Ax-1).
    Returns a numpy.ndarray of shape (c, Ax) with dtype float64.
    """
    return np.loadtxt(path, delimiter=',', dtype=np.float64)

In [50]:
parse_futhark_prices('validation_files/run_equilibrium-local-validate_solve-2-3-25-5-0-M.val')

array([[200.        , 169.2948619 , 141.47740163, 117.0506919 ,
         95.96637377,  77.96530383,  62.62372555,  49.514447  ,
         38.31752597,  28.83231542,  20.94596407,  14.59768623,
          9.74911362,   6.33721219,   4.18410726,   2.96706019,
          2.33557   ,   2.0274529 ,   1.88412023,   1.81988041,
          1.79051281,   1.7704009 ,   1.73140671,   1.5988312 ,
          1.06290795],
       [260.        , 225.25506523, 193.02633663, 163.79794474,
        137.68705753, 114.73608295,  94.842273  ,  77.72158179,
         62.98567507,  50.27381226,  39.32270211,  29.96686878,
         22.11232877,  15.70826562,  10.72367414,   7.11425982,
          4.74873215,   3.35688607,   2.60697298,   2.22655565,
          2.03958087,   1.94245189,   1.8640367 ,   1.70520958,
          1.1400604 ],
       [260.        , 225.25506523, 193.02633663, 163.79794474,
        137.68705753, 114.73608295,  94.842273  ,  77.72158179,
         62.98567507,  50.27381226,  39.32270211,  29.9668

In [51]:
parse_matlab_prices('matlab_results_for_validation/matlab-validate_solve-2-3-25-5-0.dat')

array([[200.        , 169.29486084, 141.4773999 , 117.05068984,
         95.96637165,  77.96530179,  62.62372362,  49.5144452 ,
         38.3175243 ,  28.83231391,  20.94596275,  14.59768512,
          9.7491127 ,   6.33721146,   4.18410672,   2.96705985,
          2.3355698 ,   2.02745279,   1.88412018,   1.81988038,
          1.7905128 ,   1.7704009 ,   1.73140671,   1.5988312 ,
          1.06290794],
       [260.        , 225.25506359, 193.0263338 , 163.79794114,
        137.68705352, 114.73607879,  94.84226882,  77.72157762,
         62.98567095,  50.27380827,  39.32269838,  29.96686546,
         22.11232603,  15.70826358,  10.72367279,   7.11425896,
          4.7487316 ,   3.35688573,   2.60697279,   2.22655554,
          2.03958082,   1.94245186,   1.86403669,   1.70520957,
          1.14006039],
       [260.        , 225.25506359, 193.0263338 , 163.79794114,
        137.68705352, 114.73607879,  94.84226882,  77.72157762,
         62.98567095,  50.27380827,  39.32269838,  29.9668

##### Functions for comparing Futhark and Matlab

In [52]:
_FUT_NAME_RE = re.compile(r'validate_solve-(\d+)-(\d+)-(\d+)-(\d+)-(\d+)-[A-Z]\.val$')

def compare_validation(futhark_name, rtol=1e-4, atol=1e-6,
                         val_dir='validation_files',
                         mat_dir='matlab_results_for_validation'):
      """Compare prices in a Futhark .val file against the matching MATLAB .dat file.

      Pass/fail uses numpy.allclose semantics:
          |fut - mat| <= atol + rtol * |mat|

      Parameters
      ----------
      futhark_name : str
          Filename inside `val_dir`, e.g.
          'run_equilibrium-local-validate_solve-2-7-25-5-0-C.val'.
      rtol : float, default 1e-4
          Relative tolerance (dominates for large prices).
      atol : float, default 1e-6
          Absolute tolerance floor (dominates near scrap, where prices ~ 0).
      val_dir, mat_dir : str or Path
          Directories holding the Futhark and MATLAB validation files.

      Returns
      -------
      None
          If no MATLAB file matches the parameter tuple in `futhark_name`.
      dict
          Keys:
            'within_tol'   bool   passes np.allclose(fut, mat, rtol, atol)
            'max_abs_diff' float  max |fut - mat|
            'max_rel_diff' float  max |fut - mat| / max(|mat|, atol)
            'matlab_path'  Path
      """
      m = _FUT_NAME_RE.search(futhark_name)
      if m is None:
          raise ValueError(f"Could not extract parameter tuple from: {futhark_name}")
      n, c, abar, acc0, trans = m.groups()

      matlab_name = f"matlab-validate_solve-{n}-{c}-{abar}-{acc0}-{trans}.dat"
      matlab_path = Path(mat_dir) / matlab_name
      if not matlab_path.exists():
          return None

      fut = parse_futhark_prices(Path(val_dir) / futhark_name)
      mat = parse_matlab_prices(matlab_path)

      abs_diff = np.abs(fut - mat)
      max_abs_diff = float(np.max(abs_diff))
      max_rel_diff = float(np.max(abs_diff / np.maximum(np.abs(mat), atol)))
      return {
          'within_tol':   bool(np.allclose(fut, mat, rtol=rtol, atol=atol)),
          'max_abs_diff': max_abs_diff,
          'max_rel_diff': max_rel_diff,
          'matlab_path':  matlab_path,
      }

##### Futhark Functions for parsing and summarizing validation files

In [53]:
def parse_futhark_validation(path):
      """Parse all fields from a Futhark validate_solve .val file.

      Returns a dict with:
        'prices'        ndarray  shape (c, Ax), float64
        'max_abs_ed'    float
        'stat_res'      float
        'norm_err'      float
        'min_q'         float
        'iter'          int      Newton outer iterations
        'conv'          bool     convergence flag
        'sa_iters_tot'  list[int]  per-household SA iteration totals
        'nk_iters_tot'  list[int]  per-household NK iteration totals
        'rtrips_tot'    list[int]  per-household round-trip totals
      """
      with open(path) as f:
          lines = f.read().splitlines()

      def strip(s):
          return FUT_TYPE_SUFFIX.sub('', s)

      return {
          'prices':       np.asarray(ast.literal_eval(strip(lines[0])), dtype=np.float64),
          'max_abs_ed':   float(ast.literal_eval(strip(lines[1]))),
          'stat_res':     float(ast.literal_eval(strip(lines[2]))),
          'norm_err':     float(ast.literal_eval(strip(lines[3]))),
          'min_q':        float(ast.literal_eval(strip(lines[4]))),
          'iter':         int(ast.literal_eval(strip(lines[5]))),
          'conv':         lines[6].strip() == 'true',
          'sa_iters_tot': list(ast.literal_eval(strip(lines[7]))),
          'nk_iters_tot': list(ast.literal_eval(strip(lines[8]))),
          'rtrips_tot':   list(ast.literal_eval(strip(lines[9]))),
      }

In [54]:
def summarize_futhark_run(futhark_name, rtol=1e-4, atol=1e-6,
                            val_dir='validation_files',
                            mat_dir='matlab_results_for_validation'):
      """All Futhark validation fields (except the price matrix) plus the
      MATLAB comparison verdict.

      See parse_futhark_validation for the remaining field names. Adds:
        'within_tol'    bool or None  (None if no matching MATLAB file)
        'max_abs_diff'  float or None
        'max_rel_diff'  float or None
        'matlab_path'   Path  or None
      """
      fields = parse_futhark_validation(Path(val_dir) / futhark_name)
      del fields['prices']
      cmp = compare_validation(futhark_name, rtol=rtol, atol=atol,
                               val_dir=val_dir, mat_dir=mat_dir)
      if cmp is None:
          fields.update(within_tol=None, max_abs_diff=None,
                        max_rel_diff=None, matlab_path=None)
      else:
          fields.update(cmp)
      return fields

##### Summarize all Futhark Validation Files

In [55]:
_FUT_FILE_RE = re.compile(
      r'^(?P<run_equi>[^-]+)-(?P<val_name>.+?)-validate_solve-'
      r'(?P<n>\d+)-(?P<c>\d+)-(?P<abar>\d+)-(?P<acc0>\d+)-(?P<trans>\d+)-'
      r'(?P<backend>[A-Z])\.val$'
  )

def summarize_all_futhark(rtol=1e-4, atol=1e-6,
                            val_dir='validation_files',
                            mat_dir='matlab_results_for_validation'):
      """Summarize every .val file in `val_dir`.

      Each row is a dict carrying:
        filename       str       the .val filename
        run_equi       str       e.g. 'run_equilibrium' / 'run_equilibrium_man'
        val_name       str       VAL_NAME tag, e.g. 'local' / 'default'
        n, c, abar, acc0, trans  int   parameter tuple
        backend        str       'C', 'M', 'O', 'U'
      plus everything `summarize_futhark_run` returns (prices, max_abs_ed,
      stat_res, norm_err, min_q, iter, conv, sa_iters_tot, nk_iters_tot,
      rtrips_tot, within_tol, max_abs_diff, max_rel_diff, matlab_path).

      Files whose names don't match the validate_solve pattern are skipped.
      """
      val_dir = Path(val_dir)
      int_keys = {'n', 'c', 'abar', 'acc0', 'trans'}
      rows = []
      for path in sorted(val_dir.glob('*.val')):
          m = _FUT_FILE_RE.match(path.name)
          if m is None:
              continue
          meta = {k: int(v) if k in int_keys else v
                  for k, v in m.groupdict().items()}
          meta['filename'] = path.name
          summary = summarize_futhark_run(path.name, rtol=rtol, atol=atol,
                                          val_dir=val_dir, mat_dir=mat_dir)
          rows.append({**meta, **summary})
      return rows

In [56]:
rows = summarize_all_futhark()
print(rows[0])  # Print the first row, adjust index as needed

{'run_equi': 'run_equilibrium', 'val_name': 'futhark01', 'n': 2, 'c': 1, 'abar': 25, 'acc0': 5, 'trans': 0, 'backend': 'C', 'filename': 'run_equilibrium-futhark01-validate_solve-2-1-25-5-0-C.val', 'max_abs_ed': 1.77605e-10, 'stat_res': 6e-15, 'norm_err': 0.0, 'min_q': 0.00047250047593, 'iter': 7, 'conv': True, 'sa_iters_tot': [30, 33], 'nk_iters_tot': [12, 7], 'rtrips_tot': [7, 7], 'within_tol': True, 'max_abs_diff': 2.06420328652257e-06, 'max_rel_diff': 1.0413755867201788e-07, 'matlab_path': WindowsPath('matlab_results_for_validation/matlab-validate_solve-2-1-25-5-0.dat')}


##### Benchmark parsing

In [57]:
def parse_matlab_benchmark(path):
    """Parse matlab_eqb_<variant>.dat. Columns: n c abar acc0 trans mean stdev se."""
    arr = np.loadtxt(path)
    if arr.ndim == 1:
        arr = arr[None, :]
    cols = ['n', 'c', 'abar', 'acc0', 'trans', 'mean', 'stdev', 'se']
    return [dict(zip(cols, row)) for row in arr]

def parse_futhark_benchmark(path):
    """Parse a Futhark bench_solve_<variant>.dat (or saved copy).
    Columns: n c abar acc0 trans backend mean stdev se."""
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 9:
                continue
            rows.append({
                'n':       int(parts[0]),  'c':       int(parts[1]),
                'abar':    int(parts[2]),  'acc0':    int(parts[3]),
                'trans':   int(parts[4]),  'backend': parts[5],
                'mean':    float(parts[6]),
                'stdev':   float(parts[7]),
                'se':      float(parts[8]),
            })
    return rows

In [58]:
_SAVED_BENCH_RE = re.compile(
      r'^(?P<run_equi>[^-]+)-(?P<val_name>.+?)-bench(?P<runs>\d+)-'
      r'(?P<variant>cars|households|age)\.dat$'
  )

def bench_time_table(variant, saved_dir='saved_futhark_benchmarks'):
    """Long-format DataFrame of bench mean & stdev for the given variant.
    One row per (source, run_equi, val_name, backend, parameter tuple)."""
    rows = []

    mat_path = Path(f'matlab_eqb_{variant}.dat')
    if mat_path.exists():
        for r in parse_matlab_benchmark(mat_path):
            rows.append({
                'source': 'matlab', 'run_equi': '-', 'val_name': '-',
                'backend': '-',     'runs': '-',
                'n': int(r['n']), 'c': int(r['c']), 'abar': int(r['abar']),
                'mean': r['mean'], 'stdev': r['stdev'],
            })

    for path in sorted(Path(saved_dir).glob(f'*-{variant}.dat')):
        m = _SAVED_BENCH_RE.match(path.name)
        if m is None:
            continue
        meta = m.groupdict()
        for r in parse_futhark_benchmark(path):
            rows.append({
                'source':   'futhark',
                'run_equi': meta['run_equi'],
                'val_name': meta['val_name'],
                'backend':  r['backend'],
                'runs':     int(meta['runs']),
                'n': r['n'], 'c': r['c'], 'abar': r['abar'],
                'mean': r['mean'], 'stdev': r['stdev'],
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        sort_key = {'cars': 'c', 'households': 'n', 'age': 'abar'}[variant]
        df = df.sort_values([sort_key, 'source', 'run_equi', 'val_name', 'backend'])
    return df.reset_index(drop=True)

##### Comparison Table for Validation

In [59]:
def comparison_table(variant, rtol=1e-4, atol=1e-6,
                       val_dir='validation_files',
                       mat_dir='matlab_results_for_validation'):
      """DataFrame summarising every .val file that belongs to this variant.

      Each row: one (run_equi, val_name, backend, parameter tuple) combination,
      with columns including within_tol, max_abs_diff, max_rel_diff, and the
      parsed Futhark fields (iter, conv, sa_iters_tot, nk_iters_tot, etc.).
      """
      rows = summarize_all_futhark(rtol=rtol, atol=atol,
                                   val_dir=val_dir, mat_dir=mat_dir)
      if variant == 'cars':
          keep = lambda r: r['n'] == 2 and r['abar'] == 25
          sort_key = 'c'
      elif variant == 'households':
          keep = lambda r: r['c'] == 7 and r['abar'] == 25
          sort_key = 'n'
      elif variant == 'age':
          keep = lambda r: r['n'] == 2 and r['c'] == 7
          sort_key = 'abar'
      else:
          raise ValueError(f"Unknown variant: {variant!r}")

      df = pd.DataFrame([r for r in rows if keep(r)])
      if not df.empty:
          df = (df.drop(columns=['matlab_path'], errors='ignore')
                  .sort_values(['run_equi', 'val_name', 'backend', sort_key])
                  .reset_index(drop=True))
      return df

##### Tables

In [60]:
for v in ['cars', 'households', 'age']:
    print(f"\n=== {v}: bench times ===")
    display(bench_time_table(v))

for v in ['cars', 'households', 'age']:
    print(f"\n=== {v}: MATLAB comparison + Futhark diagnostics ===")
    display(comparison_table(v))


=== cars: bench times ===


,source,run_equi,val_name,backend,runs,n,c,abar,mean,stdev
0,futhark,run_equilibrium,default,C,5,2,1,25,0.071775,0.000074
1,futhark,run_equilibrium,default,M,5,2,1,25,0.070213,0.000340
2,futhark,run_equilibrium,default,U,5,2,1,25,0.166022,0.108779
3,futhark,run_equilibrium,futhark01,C,3,2,1,25,0.071775,0.000074
4,futhark,run_equilibrium,futhark01,M,3,2,1,25,0.070213,0.000340
...,...,...,...,...,...,...,...,...,...,...
72,futhark,run_equilibrium,local,C,3,2,13,25,35.844293,0.486860
73,futhark,run_equilibrium,local,M,3,2,13,25,10.025811,0.216102
74,futhark,run_equilibrium_man,local,C,3,2,13,25,25.755916,0.351094
75,futhark,run_equilibrium_man,local,M,3,2,13,25,6.474602,0.065324



=== households: bench times ===


,source,run_equi,val_name,backend,runs,n,c,abar,mean,stdev
0,futhark,run_equilibrium,local,C,3,2,7,25,4.871783,0.024976
1,futhark,run_equilibrium,local,M,3,2,7,25,1.459303,0.074213
2,futhark,run_equilibrium_man,local,C,3,2,7,25,3.089328,0.001858
3,futhark,run_equilibrium_man,local,M,3,2,7,25,1.009094,0.051523
4,matlab,-,-,-,-,2,7,25,6.078210,0.160185
5,futhark,run_equilibrium,local,C,3,3,7,25,6.261275,0.078366
6,futhark,run_equilibrium,local,M,3,3,7,25,1.591209,0.031246
7,futhark,run_equilibrium_man,local,C,3,3,7,25,3.967121,0.096566
8,futhark,run_equilibrium_man,local,M,3,3,7,25,1.221463,0.074350
9,matlab,-,-,-,-,3,7,25,5.936394,0.003752



=== age: bench times ===


...

,run_equi,val_name,n,c,abar,acc0,trans,backend,filename,max_abs_ed,...,norm_err,min_q,iter,conv,sa_iters_tot,nk_iters_tot,rtrips_tot,within_tol,max_abs_diff,max_rel_diff
0,run_equilibrium,futhark01,2,1,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-1-2...,1.776050e-10,...,0.000000e+00,0.000473,7,True,"[30, 33]","[12, 7]","[7, 7]",True,0.000002,1.041376e-07
1,run_equilibrium,futhark01,2,3,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-3-2...,2.800000e-14,...,1.000000e-15,0.000243,8,True,"[32, 32]","[8, 8]","[8, 8]",True,0.000004,1.298642e-07
2,run_equilibrium,futhark01,2,5,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-5-2...,1.197000e-12,...,0.000000e+00,0.000165,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,2.444757e-07
3,run_equilibrium,futhark01,2,7,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-7-2...,1.170000e-13,...,0.000000e+00,0.000125,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635888e-07
4,run_equilibrium,futhark01,2,9,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-9-2...,1.601310e-10,...,1.000000e-15,0.000100,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.857850e-07
5,run_equilibrium,futhark01,2,11,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-11-...,1.488600e-11,...,0.000000e+00,0.000084,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.817315e-07
6,run_equilibrium,futhark01,2,13,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-13-...,3.035000e-12,...,2.000000e-15,0.000072,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.614684e-07
7,run_equilibrium,futhark01,2,1,25,5,0,M,run_equilibrium-futhark01-validate_solve-2-1-2...,1.776050e-10,...,0.000000e+00,0.000473,7,True,"[30, 33]","[12, 7]","[7, 7]",True,0.000002,1.041376e-07
8,run_equilibrium,futhark01,2,3,25,5,0,M,run_equilibrium-futhark01-validate_solve-2-3-2...,2.800000e-14,...,1.000000e-15,0.000243,8,True,"[32, 32]","[8, 8]","[8, 8]",True,0.000004,1.298642e-07
9,run_equilibrium,futhark01,2,5,25,5,0,M,run_equilibrium-futhark01-validate_solve-2-5-2...,1.197000e-12,...,0.000000e+00,0.000165,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,2.444757e-07



=== households: MATLAB comparison + Futhark diagnostics ===


,run_equi,val_name,n,c,abar,acc0,trans,backend,filename,max_abs_ed,...,norm_err,min_q,iter,conv,sa_iters_tot,nk_iters_tot,rtrips_tot,within_tol,max_abs_diff,max_rel_diff
0,run_equilibrium,futhark01,2,7,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-7-2...,1.170000e-13,...,0.000000e+00,0.000125,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635888e-07
1,run_equilibrium,futhark01,2,7,25,5,0,M,run_equilibrium-futhark01-validate_solve-2-7-2...,1.170000e-13,...,0.000000e+00,0.000125,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635888e-07
2,run_equilibrium,futhark01,2,7,25,5,0,U,run_equilibrium-futhark01-validate_solve-2-7-2...,3.100000e-14,...,0.000000e+00,0.000125,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635882e-07
3,run_equilibrium,local,2,7,25,5,0,C,run_equilibrium-local-validate_solve-2-7-25-5-...,9.000000e-14,...,0.000000e+00,0.000125,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635894e-07
4,run_equilibrium,local,3,7,25,5,0,C,run_equilibrium-local-validate_solve-3-7-25-5-...,1.450000e-13,...,0.000000e+00,0.000273,6,True,"[24, 24, 24]","[6, 6, 6]","[6, 6, 6]",True,0.000007,3.291829e-07
5,run_equilibrium,local,4,7,25,5,0,C,run_equilibrium-local-validate_solve-4-7-25-5-...,1.070000e-13,...,0.000000e+00,0.000384,6,True,"[24, 24, 24, 24]","[6, 6, 6, 6]","[6, 6, 6, 6]",True,0.000005,2.529961e-07
6,run_equilibrium,local,5,7,25,5,0,C,run_equilibrium-local-validate_solve-5-7-25-5-...,6.869820e-10,...,0.000000e+00,0.000457,5,True,"[20, 20, 20, 20, 20]","[5, 5, 5, 5, 5]","[5, 5, 5, 5, 5]",True,0.000006,2.735985e-07
7,run_equilibrium,local,6,7,25,5,0,C,run_equilibrium-local-validate_solve-6-7-25-5-...,5.542000e-10,...,0.000000e+00,0.000505,5,True,"[20, 20, 20, 20, 20, 20]","[5, 5, 5, 5, 5, 5]","[5, 5, 5, 5, 5, 5]",True,0.000006,2.805694e-07
8,run_equilibrium,local,2,7,25,5,0,M,run_equilibrium-local-validate_solve-2-7-25-5-...,9.000000e-14,...,0.000000e+00,0.000125,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635894e-07
9,run_equilibrium,local,3,7,25,5,0,M,run_equilibrium-local-validate_solve-3-7-25-5-...,1.450000e-13,...,0.000000e+00,0.000273,6,True,"[24, 24, 24]","[6, 6, 6]","[6, 6, 6]",True,0.000007,3.291829e-07



=== age: MATLAB comparison + Futhark diagnostics ===


,run_equi,val_name,n,c,abar,acc0,trans,backend,filename,max_abs_ed,...,norm_err,min_q,iter,conv,sa_iters_tot,nk_iters_tot,rtrips_tot,within_tol,max_abs_diff,max_rel_diff
0,run_equilibrium,futhark01,2,7,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-7-2...,1.170000e-13,...,0.000000e+00,1.245906e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635888e-07
1,run_equilibrium,futhark01,2,7,25,5,0,M,run_equilibrium-futhark01-validate_solve-2-7-2...,1.170000e-13,...,0.000000e+00,1.245906e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635888e-07
2,run_equilibrium,futhark01,2,7,25,5,0,U,run_equilibrium-futhark01-validate_solve-2-7-2...,3.100000e-14,...,0.000000e+00,1.245906e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635882e-07
3,run_equilibrium,local,2,7,10,5,0,C,run_equilibrium-local-validate_solve-2-7-10-5-...,2.167800e-11,...,1.000000e-15,9.404310e-03,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000002,4.316706e-08
4,run_equilibrium,local,2,7,15,5,0,C,run_equilibrium-local-validate_solve-2-7-15-5-...,4.400000e-14,...,0.000000e+00,8.037581e-03,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000005,1.926050e-07
5,run_equilibrium,local,2,7,20,5,0,C,run_equilibrium-local-validate_solve-2-7-20-5-...,8.180000e-12,...,1.000000e-15,1.622109e-03,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.662702e-07
6,run_equilibrium,local,2,7,25,5,0,C,run_equilibrium-local-validate_solve-2-7-25-5-...,9.000000e-14,...,0.000000e+00,1.245906e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635894e-07
7,run_equilibrium,local,2,7,30,5,0,C,run_equilibrium-local-validate_solve-2-7-30-5-...,4.000000e-14,...,0.000000e+00,1.013898e-05,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000007,2.794741e-07
8,run_equilibrium,local,2,7,35,5,0,C,run_equilibrium-local-validate_solve-2-7-35-5-...,9.900000e-14,...,1.000000e-15,8.315462e-07,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000007,2.799861e-07
9,run_equilibrium,local,2,7,10,5,0,M,run_equilibrium-local-validate_solve-2-7-10-5-...,2.167800e-11,...,1.000000e-15,9.404310e-03,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000002,4.316706e-08


##### LaTeX Table

In [61]:
def local_bench_table(variant, val_name='local'):
      """Wide table for `variant`: rows = varying parameter (c / n / abar),
      columns = (MATLAB / Futhark-C / Futhark-M) × (mean, stdev).
      Only Futhark rows whose val_name matches `val_name` are kept."""
      df = bench_time_table(variant)

      mask = (df['source'] == 'matlab') | (
          (df['source'] == 'futhark') & (df['val_name'] == val_name)
      )
      df = df[mask].copy()
      df['label'] = df.apply(
          lambda r: 'MATLAB' if r['source'] == 'matlab' else f"Futhark-{r['backend']}",
          axis=1,
      )

      var_col = {'cars': 'c', 'households': 'n', 'age': 'abar'}[variant]
      pivot = df.pivot_table(index=var_col, columns='label',
                             values=['mean', 'stdev'], aggfunc='first')
      pivot = pivot.swaplevel(0, 1, axis=1)

      label_order = ['MATLAB', 'Futhark-C', 'Futhark-M']
      present = [l for l in label_order if l in pivot.columns.get_level_values(0).unique()]
      cols = [(lbl, stat) for lbl in present for stat in ['mean', 'stdev']]
      return pivot.reindex(columns=cols)

In [62]:
for v in ['cars', 'households', 'age']:
      print(f"\n=== {v}: bench times (local) ===")
      tbl = local_bench_table(v)
      display(tbl)
      print(tbl.to_latex(float_format='%.4f',
                         caption=f'Equilibrium benchmark times ({v})',
                         label=f'tab:bench-{v}'))


=== cars: bench times (local) ===


label     MATLAB            Futhark-C            Futhark-M          
            mean     stdev       mean     stdev       mean     stdev
c                                                                   
1       0.060373  0.002757   0.015463  0.000099   0.030144  0.002961
3       0.406809  0.025374   0.396969  0.003845   0.165419  0.005293
5       1.712243  0.171639   1.651963  0.039652   0.494155  0.027531
7       6.339220  0.232967   4.871783  0.024976   1.459303  0.074213
9       8.990580  0.535143  10.226384  0.031918   2.758508  0.032983
11     16.764712  0.111942  19.627541  0.237127   5.381744  0.128018
13     27.687078  0.490121  35.844293  0.486860  10.025811  0.216102

\begin{table}
\caption{Equilibrium benchmark times (cars)}
\label{tab:bench-cars}
\begin{tabular}{lrrrrrr}
\toprule
label & \multicolumn{2}{r}{MATLAB} & \multicolumn{2}{r}{Futhark-C} & \multicolumn{2}{r}{Futhark-M} \\
 & mean & stdev & mean & stdev & mean & stdev \\
c &  &  &  &  &  &  \\
\midrule
1 & 0.0604 & 0.0028 & 0.0155 & 0.0001 & 0.0301 & 0.0030 \\
3 & 0.4068 & 0.0254 & 0.3970 & 0.0038 & 0.1654 & 0.0053 \\
5 & 1.7122 & 0.1716 & 1.6520 & 0.0397 & 0.4942 & 0.0275 \\
7 & 6.3392 & 0.2330 & 4.8718 & 0.0250 & 1.4593 & 0.0742 \\
9 & 8.9906 & 0.5351 & 10.2264 & 0.0319 & 2.7585 & 0.0330 \\
11 & 16.7647 & 0.1119 & 19.6275 & 0.2371 & 5.3817 & 0.1280 \\
13 & 27.6871 & 0.4901 & 35.8443 & 0.4869 & 10.0258 & 0.2161 \\
\bottomrule
\end{tabular}
\end{table}


=== households: bench times (local) ===


label    MATLAB            Futhark-C           Futhark-M          
           mean     stdev       mean     stdev      mean     stdev
n                                                                 
2      6.078210  0.160185   4.871783  0.024976  1.459303  0.074213
3      5.936394  0.003752   6.261275  0.078366  1.591209  0.031246
4      7.193715  0.007653   8.121412  0.143377  2.061760  0.026745
5      9.802116  0.029087   8.346185  0.083159  2.140306  0.056007
6      9.763269  0.008041  10.233034  0.226026  2.519597  0.075724

\begin{table}
\caption{Equilibrium benchmark times (households)}
\label{tab:bench-households}
\begin{tabular}{lrrrrrr}
\toprule
label & \multicolumn{2}{r}{MATLAB} & \multicolumn{2}{r}{Futhark-C} & \multicolumn{2}{r}{Futhark-M} \\
 & mean & stdev & mean & stdev & mean & stdev \\
n &  &  &  &  &  &  \\
\midrule
2 & 6.0782 & 0.1602 & 4.8718 & 0.0250 & 1.4593 & 0.0742 \\
3 & 5.9364 & 0.0038 & 6.2613 & 0.0784 & 1.5912 & 0.0312 \\
4 & 7.1937 & 0.0077 & 8.1214 & 0.1434 & 2.0618 & 0.0267 \\
5 & 9.8021 & 0.0291 & 8.3462 & 0.0832 & 2.1403 & 0.0560 \\
6 & 9.7633 & 0.0080 & 10.2330 & 0.2260 & 2.5196 & 0.0757 \\
\bottomrule
\end{tabular}
\end{table}


=== age: bench times (local) ===


label     MATLAB            Futhark-C           Futhark-M          
            mean     stdev       mean     stdev      mean     stdev
abar                                                               
10      0.622916  0.019175   0.325681  0.000760  0.118980  0.002272
15      0.971457  0.015242   0.906537  0.016835  0.251624  0.006229
20      2.491372  0.055827   2.220675  0.034330  0.526127  0.010412
25      6.115346  0.211530   4.871783  0.024976  1.459303  0.074213
30     10.537881  0.243166   9.599313  0.198846  2.909811  0.214490
35     16.424521  0.603839  16.559450  0.696340  4.494586  0.120568

\begin{table}
\caption{Equilibrium benchmark times (age)}
\label{tab:bench-age}
\begin{tabular}{lrrrrrr}
\toprule
label & \multicolumn{2}{r}{MATLAB} & \multicolumn{2}{r}{Futhark-C} & \multicolumn{2}{r}{Futhark-M} \\
 & mean & stdev & mean & stdev & mean & stdev \\
abar &  &  &  &  &  &  \\
\midrule
10 & 0.6229 & 0.0192 & 0.3257 & 0.0008 & 0.1190 & 0.0023 \\
15 & 0.9715 & 0.0152 & 0.9065 & 0.0168 & 0.2516 & 0.0062 \\
20 & 2.4914 & 0.0558 & 2.2207 & 0.0343 & 0.5261 & 0.0104 \\
25 & 6.1153 & 0.2115 & 4.8718 & 0.0250 & 1.4593 & 0.0742 \\
30 & 10.5379 & 0.2432 & 9.5993 & 0.1988 & 2.9098 & 0.2145 \\
35 & 16.4245 & 0.6038 & 16.5595 & 0.6963 & 4.4946 & 0.1206 \\
\bottomrule
\end{tabular}
\end{table}



##### Validation Tables

In [71]:
def _pivot_validation(variant, rename, val_name='local'):
      """Pivot validation rows for `variant`, keeping only the metrics in `rename`."""
      rows = comparison_table(variant)
      rows = rows[rows['val_name'] == val_name].copy()
      if rows.empty:
          return pd.DataFrame()

      metrics = list(rename)

      # Compact list-valued counters: [30, 30] -> '30';  [30, 33] -> '30,33'
      def compact(lst):
          return str(lst[0]) if len(set(lst)) == 1 else ','.join(map(str, lst))
      for col in {'sa_iters_tot', 'nk_iters_tot', 'rtrips_tot'} & set(metrics):
          rows[col] = rows[col].apply(compact)

      var_col = {'cars': 'c', 'households': 'n', 'age': 'abar'}[variant]
      rows['label'] = rows['run_equi'] + '-' + rows['backend']

      indexed = rows.set_index([var_col, 'label'])[metrics]
      pivot = indexed.unstack('label').rename(columns=rename, level=0)
      pivot = pivot.swaplevel(0, 1, axis=1)

      label_order = sorted(rows['label'].unique())
      cols = [(lbl, rename[m]) for lbl in label_order for m in metrics]
      return pivot.reindex(columns=cols)


def local_validation_summary(variant, val_name='local'):
    """Newton / Conv / Match per (run_equi, backend)."""
    return _pivot_validation(variant,
        {'iter': 'Newton', 'conv': 'Conv', 'within_tol': 'Match'}, val_name)


def local_validation_iters(variant, val_name='local'):
    """SA / NK / RT per (run_equi, backend)."""
    return _pivot_validation(variant,
        {'sa_iters_tot': 'SA', 'nk_iters_tot': 'NK', 'rtrips_tot': 'RT'}, val_name)

In [72]:
for v in ['cars', 'households', 'age']:
      print(f"\n=== {v}: convergence summary (local) ===")
      tbl = local_validation_summary(v)
      display(tbl)
      print(tbl.to_latex(caption=f'Convergence summary ({v})',
                         label=f'tab:val-summary-{v}'))

      print(f"\n=== {v}: iteration counts (local) ===")
      tbl = local_validation_iters(v)
      display(tbl)
      print(tbl.to_latex(caption=f'Iteration counts ({v})',
                         label=f'tab:val-iters-{v}'))


=== cars: convergence summary (local) ===


label run_equilibrium-C             run_equilibrium-M              \
                 Newton  Conv Match            Newton  Conv Match   
c                                                                   
1                     7  True  True                 7  True  True   
3                     8  True  True                 8  True  True   
5                     7  True  True                 7  True  True   
7                     7  True  True                 7  True  True   
9                     6  True  True                 6  True  True   
11                    6  True  True                 6  True  True   
13                    6  True  True                 6  True  True   

label run_equilibrium_man-C             run_equilibrium_man-M              
                     Newton  Conv Match                Newton  Conv Match  
c                                                                          
1                         7  True  True                     7  True  True  
3                         8  True  True                     8  True  True  
5                         7  True  True                     7  True  True  
7                         7  True  True                     7  True  True  
9                         6  True  True                     6  True  True  
11                        6  True  True                     6  True  True  
13                        6  True  True                     6  True  True

\begin{table}
\caption{Convergence summary (cars)}
\label{tab:val-summary-cars}
\begin{tabular}{lrrrrrrrrrrrr}
\toprule
label & \multicolumn{3}{r}{run_equilibrium-C} & \multicolumn{3}{r}{run_equilibrium-M} & \multicolumn{3}{r}{run_equilibrium_man-C} & \multicolumn{3}{r}{run_equilibrium_man-M} \\
 & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match \\
c &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
1 & 7 & True & True & 7 & True & True & 7 & True & True & 7 & True & True \\
3 & 8 & True & True & 8 & True & True & 8 & True & True & 8 & True & True \\
5 & 7 & True & True & 7 & True & True & 7 & True & True & 7 & True & True \\
7 & 7 & True & True & 7 & True & True & 7 & True & True & 7 & True & True \\
9 & 6 & True & True & 6 & True & True & 6 & True & True & 6 & True & True \\
11 & 6 & True & True & 6 & True & True & 6 & True & True & 6 & True & True \\
13 & 6 & True & True & 6 & True & True & 6 & True & True & 6 & True & True \\
\bottomrule


label run_equilibrium-C          run_equilibrium-M           \
                     SA    NK RT                SA    NK RT   
c                                                             
1                 30,33  12,7  7             30,33  12,7  7   
3                    32     8  8                32     8  8   
5                    28     7  7                28     7  7   
7                    28     7  7                28     7  7   
9                    24     6  6                24     6  6   
11                   24     6  6                24     6  6   
13                   24     6  6                24     6  6   

label run_equilibrium_man-C          run_equilibrium_man-M           
                         SA    NK RT                    SA    NK RT  
c                                                                    
1                     30,33  12,7  7                 30,33  12,7  7  
3                        32     8  8                    32     8  8  
5                        28     7  7                    28     7  7  
7                        28     7  7                    28     7  7  
9                        24     6  6                    24     6  6  
11                       24     6  6                    24     6  6  
13                       24     6  6                    24     6  6

\begin{table}
\caption{Iteration counts (cars)}
\label{tab:val-iters-cars}
\begin{tabular}{lllllllllllll}
\toprule
label & \multicolumn{3}{r}{run_equilibrium-C} & \multicolumn{3}{r}{run_equilibrium-M} & \multicolumn{3}{r}{run_equilibrium_man-C} & \multicolumn{3}{r}{run_equilibrium_man-M} \\
 & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT \\
c &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
1 & 30,33 & 12,7 & 7 & 30,33 & 12,7 & 7 & 30,33 & 12,7 & 7 & 30,33 & 12,7 & 7 \\
3 & 32 & 8 & 8 & 32 & 8 & 8 & 32 & 8 & 8 & 32 & 8 & 8 \\
5 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 \\
7 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 \\
9 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 \\
11 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 \\
13 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 \\
\bottomrule
\end{tabular}
\end{table}


=== households: convergence summary (local) ===


...

\begin{table}
\caption{Iteration counts (households)}
\label{tab:val-iters-households}
\begin{tabular}{lllllllllllll}
\toprule
label & \multicolumn{3}{r}{run_equilibrium-C} & \multicolumn{3}{r}{run_equilibrium-M} & \multicolumn{3}{r}{run_equilibrium_man-C} & \multicolumn{3}{r}{run_equilibrium_man-M} \\
 & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT \\
n &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
2 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 \\
3 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 \\
4 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 \\
5 & 20 & 5 & 5 & 20 & 5 & 5 & 20 & 5 & 5 & 20 & 5 & 5 \\
6 & 20 & 5 & 5 & 20 & 5 & 5 & 20 & 5 & 5 & 20 & 5 & 5 \\
\bottomrule
\end{tabular}
\end{table}


=== age: convergence summary (local) ===


label run_equilibrium-C             run_equilibrium-M              \
                 Newton  Conv Match            Newton  Conv Match   
abar                                                                
10                    6  True  True                 6  True  True   
15                    6  True  True                 6  True  True   
20                    6  True  True                 6  True  True   
25                    7  True  True                 7  True  True   
30                    7  True  True                 7  True  True   
35                    7  True  True                 7  True  True   

label run_equilibrium_man-C             run_equilibrium_man-M              
                     Newton  Conv Match                Newton  Conv Match  
abar                                                                       
10                        6  True  True                     6  True  True  
15                        6  True  True                     6  True  True  
20                        6  True  True                     6  True  True  
25                        7  True  True                     7  True  True  
30                        7  True  True                     7  True  True  
35                        7  True  True                     7  True  True

\begin{table}
\caption{Convergence summary (age)}
\label{tab:val-summary-age}
\begin{tabular}{lrrrrrrrrrrrr}
\toprule
label & \multicolumn{3}{r}{run_equilibrium-C} & \multicolumn{3}{r}{run_equilibrium-M} & \multicolumn{3}{r}{run_equilibrium_man-C} & \multicolumn{3}{r}{run_equilibrium_man-M} \\
 & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match \\
abar &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
10 & 6 & True & True & 6 & True & True & 6 & True & True & 6 & True & True \\
15 & 6 & True & True & 6 & True & True & 6 & True & True & 6 & True & True \\
20 & 6 & True & True & 6 & True & True & 6 & True & True & 6 & True & True \\
25 & 7 & True & True & 7 & True & True & 7 & True & True & 7 & True & True \\
30 & 7 & True & True & 7 & True & True & 7 & True & True & 7 & True & True \\
35 & 7 & True & True & 7 & True & True & 7 & True & True & 7 & True & True \\
\bottomrule
\end{tabular}
\end{table}


=== age: iteration counts (local) ===


label run_equilibrium-C       run_equilibrium-M       run_equilibrium_man-C  \
                     SA NK RT                SA NK RT                    SA   
abar                                                                          
10                   24  6  6                24  6  6                    24   
15                   24  6  6                24  6  6                    24   
20                   24  6  6                24  6  6                    24   
25                   28  7  7                28  7  7                    28   
30                   28  7  7                28  7  7                    28   
35                   28  7  7                28  7  7                    28   

label       run_equilibrium_man-M        
      NK RT                    SA NK RT  
abar                                     
10     6  6                    24  6  6  
15     6  6                    24  6  6  
20     6  6                    24  6  6  
25     7  7                    28  7  7  
30     7  7                    28  7  7  
35     7  7                    28  7  7

\begin{table}
\caption{Iteration counts (age)}
\label{tab:val-iters-age}
\begin{tabular}{lllllllllllll}
\toprule
label & \multicolumn{3}{r}{run_equilibrium-C} & \multicolumn{3}{r}{run_equilibrium-M} & \multicolumn{3}{r}{run_equilibrium_man-C} & \multicolumn{3}{r}{run_equilibrium_man-M} \\
 & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT \\
abar &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
10 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 \\
15 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 \\
20 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 & 24 & 6 & 6 \\
25 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 \\
30 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 \\
35 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 & 28 & 7 & 7 \\
\bottomrule
\end{tabular}
\end{table}

